In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression, Ridge, RidgeCV
from IPython.display import display

In [11]:
data =  'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
!curl -O {data}

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100 272.2k 100 272.2k   0      0 404.1k      0                              0
100 272.2k 100 272.2k   0      0 404.0k      0                              0
100 272.2k 100 272.2k   0      0 404.0k      0                              0


In [12]:
df = pd.read_csv("course_lead_scoring_2026.csv")

In [16]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [17]:
df.columns = df.columns.str.lower().str.replace(" ", "_")
categorical_cols = df.select_dtypes(include=['str', 'object']).columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

In [18]:
df[categorical_cols] = df[categorical_cols].fillna("NA")
df[numerical_cols] = df[numerical_cols].fillna(0.0)

In [19]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [25]:
#Question 1
#What is the most frequent observation (mode) for the column industry?

   # NA
   # technology
   # healthcare
   # retail
df.industry.value_counts(ascending=False)

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [28]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

y_train = df_train.converted.values
y_test = df_test.converted.values
y_val = df_val.converted.values

del df_train['converted']
del df_test['converted']
del df_val['converted']


In [ ]:
'''Question 2
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

    interaction_count and lead_score: 0.916626
    number_of_courses_viewed and lead_score: 0.757178 
    number_of_courses_viewed and interaction_count: 0.723040
    annual_income and interaction_count: 0.122528

Only consider the pairs above when answering this question.'''

corr_matrix_df_full_train = df_full_train.corr(numeric_only=True)
print(corr_matrix_df_full_train)

                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.163680   
number_of_courses_viewed       0.163680                  1.000000   
interaction_count              0.122528                  0.723040   
lead_score                     0.222493                  0.757178   
converted                      0.188437                  0.393200   

                          interaction_count  lead_score  converted  
annual_income                      0.122528    0.222493   0.188437  
number_of_courses_viewed           0.723040    0.757178   0.393200  
interaction_count                  1.000000    0.916626   0.450273  
lead_score                         0.916626    1.000000   0.482078  
converted                          0.450273    0.482078   1.000000  


In [ ]:
'''Question 3

    Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
    Round the scores to 2 decimals using round(score, 2).

Which of these variables has the biggest mutual information score?

lead_source          0.034761
employment_status    0.020313
industry             0.002898
location             0.001050
'''
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.converted)


mi_df_full_train = df_full_train[categorical_cols].apply(mutual_info_churn_score)
mi_df_full_train.sort_values(ascending=False)

lead_source          0.034761
employment_status    0.020313
industry             0.002898
location             0.001050
dtype: float64

In [35]:
'''Question 4

    Now let's train a logistic regression.
    Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
    Fit the model on the training dataset.
        To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
        model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

    0.55
    0.65 ***
    0.75
    0.85
'''

dv = DictVectorizer(sparse=False)

train_dict = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val.to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:,1]
converted_decision = (y_pred >= 0.5)
round((y_val == converted_decision).mean(),2)

np.float64(0.64)

In [ ]:
'''Question 5

    Let's find the least useful feature using the feature elimination technique.
    Train a model using the same features and parameters as in Q4 (without rounding).
    Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
    For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

    'lead_source'
    'number_of_courses_viewed'
    'interaction_count'

    Note: The difference doesn't have to be positive.
    ** Orignal from # 4: 0.645
    lead_source 0.642 0.0030000000000000027
interaction_count 0.601 0.04400000000000004
number_of_courses_viewed 0.643 0.0020000000000000018
'''
def eval_features (xtrain, xval, ytrain):
    dv = DictVectorizer(sparse=False)

    train_dict = xtrain.to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    val_dict = xval.to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, ytrain)

    y_pred = model.predict_proba(X_val)[:,1]
    converted_decision = (y_pred >= 0.5)
    return (y_val == converted_decision).mean()

for i in {'lead_source', 'number_of_courses_viewed', 'interaction_count'}:
    t_xtrain = df_train.copy()
    del t_xtrain[i]
    t_xval = df_val.copy()
    del t_xval[i]
    t_xtest = df_test.copy()
    del t_xtest[i]
    #print (t_xtrain.head())
    
    acc = eval_features(t_xtrain, t_xval, y_train)
    print(i, acc, 0.645 - acc )

lead_source 0.642 0.0030000000000000027
interaction_count 0.601 0.04400000000000004
number_of_courses_viewed 0.643 0.0020000000000000018


In [ ]:
'''Question 6

    Now let's train a regularized logistic regression.
    Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
    Train models using all the features as in Q4.
    Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these C leads to the best accuracy on the validation set?

    0.000001
    0.00001
    0.0001
    0.001

    Note: If there are multiple options, select the smallest C.
    
    results:
    0.000001 0.598
    0.00001  0.598
    0.0001   0.613
    0.001    0.645
'''

C = [0.000001, 0.00001, 0.0001, 0.001]

dv = DictVectorizer(sparse=False)

train_dict = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val.to_dict(orient='records')
X_val = dv.transform(val_dict)

for c in C:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:,1]
    converted_decision = (y_pred >= 0.5)
    print(c,round((y_val == converted_decision).mean(),3))
    

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
0.1 0.645
